### Import Libraries

In [1]:
import os
from sentence_transformers import SentenceTransformer
import torch
import numpy as np
import pandas as pd
import spacy
import gc
import re
import tqdm
import faiss
import pickle
import json

/home/abdulrahimfami/Desktop/final_poject/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-15 16:06:05.125459: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
class VectorBaseBuilder:
    
    def __init__(self):

        self.embedder = None
        self.nlp = None
        self.df = pd.read_csv(r"data/enterprise-attack.csv")
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.data = {}
        self.ROOT = r"Knowledge_base"



    def load_models(self):
        self.embedder = SentenceTransformer(r"models/ATTACK-BERT/",device=self.device)
        self.nlp = spacy.load("en_core_web_sm",disable=["ner", "tagger", "lemmatizer", "attribute_ruler"])


    def create_embeddings(self,chunks):
        
        embeddings = []

        for chunk_record in tqdm.tqdm(chunks.values(),desc='Creating Embeddings'):
            chunk = chunk_record['chunk']
            embedding = self.embedder.encode(chunk,convert_to_numpy=True,normalize_embeddings=True,device=self.device)
            embeddings.append(embedding)

        embeddings = np.vstack(embeddings).astype('float32')
        return embeddings


    def build_docs(self):
        records = {}
        
        for ind,row in tqdm.tqdm(self.df.iterrows(),desc="Building Docs"):
            id = row['id']
            text = f"{row['name']} : {row['description']} {row['detection']}"
            records[id] = {'text':text,'metadata':{'id':id,'name':row['name']}}

        return records
    

    def create_chunks(self,records):
        chunks = {}
        tracker = 0
        
        for row in tqdm.tqdm(records.values(),desc='Creating Chunks'):
            text = row['text']
            text = re.sub(r"\s*\(Citation:[^)]+\)", "", text)

            for sent in self.nlp(text).sents:
                chunks[tracker] = {'chunk':sent.text,'metadata':{'id':row['metadata']['id'],'name':row['metadata']['name']}}
                tracker = tracker+1

        return chunks



    def build_vector_base(self):
        
        self.load_models()
        records = self.build_docs()
        chunks = self.create_chunks(records=records)
        embeddings = self.create_embeddings(chunks=chunks)
        
        
        os.makedirs(self.ROOT,exist_ok=True)
        
        with open(f"{self.ROOT}/mitre_metadata.json","w") as f:
            json.dump(chunks,f,indent=4)
        
        with open(f"{self.ROOT}/mitre_lookup.json","w") as f:
            json.dump(records,f,indent=4)
        
        dim = embeddings.shape[1]
        faiss_index = faiss.IndexFlatIP(dim)
        faiss_index.add(embeddings)
        faiss.write_index(faiss_index, f"{self.ROOT}/mitre_index.faiss")

        print("Vectorbase Has Been Created Successfully.")
        print("Total Vector Entries",faiss_index.ntotal)       
        

        


builder = VectorBaseBuilder()


In [6]:
builder.build_vector_base()

Building Docs: 566it [00:00, 23062.19it/s]
Creating Embeddings: 100%|██████████| 6472/6472 [01:02<00:00, 104.22it/s]


Vectorbase Has Been Created Successfully.
Total Vector Entries 6472


({0: {'chunk': 'Data Obfuscation : Adversaries may obfuscate command and control traffic to make it more difficult to detect.',
   'metadata': {'id': 'T1001', 'name': 'Data Obfuscation'}},
  1: {'chunk': 'Command and control (C2) communications are hidden (but not necessarily encrypted) in an attempt to make the content more difficult to discover or decipher and to make the communication less conspicuous and hide commands from being seen.',
   'metadata': {'id': 'T1001', 'name': 'Data Obfuscation'}},
  2: {'chunk': 'This encompasses many methods, such as adding junk data to protocol traffic, using steganography, or impersonating legitimate protocols.  ',
   'metadata': {'id': 'T1001', 'name': 'Data Obfuscation'}},
  3: {'chunk': 'Analyze network data for uncommon data flows (e.g., a client sending significantly more data than it receives from a server).',
   'metadata': {'id': 'T1001', 'name': 'Data Obfuscation'}},
  4: {'chunk': 'Processes utilizing the network that do not normally ha

In [96]:
import gc

def clear_memory():

    print("Allocated:", torch.cuda.memory_allocated() / 1024**3, "GB")
    print("Reserved :", torch.cuda.memory_reserved() / 1024**3, "GB")
    print("Total GPU:", torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

    gc.collect()
    torch.cuda.empty_cache()

clear_memory()

Allocated: 1.6341538429260254 GB
Reserved : 1.78125 GB
Total GPU: 3.62762451171875 GB
